In [ ]:
#https://www.kaggle.com/nicapotato/universal-sentence-encoder-semantic-similarity

In [ ]:
import os
import re
import sys
import numpy as np
import pandas as pd
rootfold = %pwd
rootfold

In [ ]:
sys.path.append(rootfold+"/../2_findflags/all_subreddits")
import functions_pos_match as fpm

In [ ]:
from importlib import reload  
reload(fpm)

### Test specific phrases

In [ ]:
phrases = ['Removed due to fox news being an unreliable source.',
           'Fox News is an unreliable source.',
           'STOP spreading misinformation and panic.',
           'Ahhh yes, more propaganda.',
           'Unreliable source - of course left this out:',
           'Clickbait bullshit instead of investigative journalism.',
           'Sloppy, bullshit headline, CTV. ',
           'author spreads fake news',
           'Do not fall for propaganda',
           'BS title']

In [ ]:
[(t.lower_, t.lemma_, t.pos_, t.dep_)  for d in phrases for t in fpm.nlp(d)]

In [ ]:
for sent_index,sent in enumerate(fpm.nlp.pipe(phrases, n_process=10, n_threads = 20)):
    currentmatch = [(sent_index, sent.text, m ) for m in sent[:]._.structurematched]
    print(currentmatch)

### Test manually classified phrases

In [ ]:
MANUAL = pd.read_csv("../4_check/data/test_classified.csv")
MANUAL["class3"]=MANUAL["class"]
MANUAL.head()

In [ ]:
disinfoFN = MANUAL.query('disinformation==1 & POSflag==0 & class3 == "f"')
for text in fpm.nlp.pipe(disinfoFN.comm_body, n_process=10, n_threads = 20, disable = ["ner"]):
    print(text)
    print('----------------------\n')
    for sent in text.sents:
        print(sent)
        currentmatch = [(sent.text, m) for m in sent[:]._.structurematched]
        print(currentmatch)
    print('\n\n')

In [ ]:
newFLAG = []
for text in fpm.nlp.pipe(MANUAL.comm_body, n_process=10, n_threads = 20, disable = ["ner"]):
    hit=0
    for sent in text.sents:
        currentmatch = [(sent.text, m) for m in sent[:]._.structurematched]
        hit+=len(currentmatch)
    newFLAG+=[hit]

In [ ]:
MANUAL['newFLAG'] = newFLAG
MANUAL.newFLAG.value_counts()

In [ ]:
for i, m in MANUAL.iterrows():
    if m.newFLAG == 0 and m.class3 =="f":
        print(m.comm_body)
        print('\n')

In [ ]:
for i, m in MANUAL.iterrows():
    if m.newFLAG == 0 and m.POSflag>0:
        print(m.comm_body)
        print('\n')

In [ ]:
for i, m in MANUAL.iterrows():
    if m.newFLAG > 0 and m.POSflag==0:
        print(m.comm_body)
        print('\n')

In [ ]:
MANUAL[['id','index','newFLAG']].to_csv("../4_check/data/check_newFLAG.csv", index = False)

### Older

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

#module_url = "https://tfhub.dev/google/universal-sentence-encoder/4" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
module_url = rootfold+"/model/universal-sentence-encoder_4"
ebmeddingmodel = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
  return ebmeddingmodel(input)
def tf_simil(s1,s2): return np.inner(s1,s2)
def tf_simil_matrix(embeddings, rd = 8): return [[round(tf_simil(s1,s2),rd) for s1 in embeddings] for s2 in embeddings]

In [ ]:
sentfiles = sorted([st for st in os.listdir(rootfold+'/../output/') if re.search('MATCH_2020_',st)])

In [ ]:
flagtypes = ['disinformation','fakenews','bs','misleading','unreliable','propaganda']

In [ ]:
allsents=pd.DataFrame()
existing_docids = set()
mean_flag_embeddings = dict()
for file in sentfiles:
    SENT = pd.read_csv(rootfold+'/../output/'+file)
    allsents = allsents.append(SENT, ignore_index = True)
    existing_docids = existing_docids.union(set(SENT.comm_id))
for flgtype in flagtypes:
    s = allsents[allsents[flgtype]>0]
    mean_flag_embeddings[flgtype] = tf.reduce_mean(embed(s[s.flag>0].sent), axis = 0)

In [ ]:
SENT.sent = SENT.sent.str.replace('’','\'')

In [ ]:
allsents2 = allsents.copy()
allsents2.sent = allsents2.sent.astype('str').str.lower().str.replace('’',"'")

In [ ]:
len(allsents2)

### Re-evaluate POS in SENTS

In [ ]:
del sys.modules["functions_pos_match"]
import functions_pos_match as fpm

In [ ]:
allsents.head()

In [ ]:
allmatches=[]
for sent_index, sent in enumerate(fpm.nlp.pipe(allsents2.sent, n_process=10, n_threads = 20, disable = ["ner"])):
    currentmatch = [(sent_index, sent.text, m ) for m in sent[:]._.structurematched]
    allmatches += currentmatch
    if sent_index % 50000 == 1:
        print('%d extracted / %d index' % (len(allmatches), sent_index))

In [ ]:
allmatches_df = pd.DataFrame(allmatches, columns = ['sent_index', 'sent', 'matchtype' ])
#allmatches_df['sent'] = [s.text for s in allmatches_df.sent]
allmatches_df.to_csv(rootfold+'/../output/comm_matches_06_02.csv')
allmatches_df.shape

New classification

In [ ]:
# Only if not rerun
allmatches_df = pd.read_csv(rootfold+'/../output/comm_matches_06_02.csv')
allmatches_df

In [ ]:
freq_matches = allmatches_df.matchtype.value_counts()
freq_matches

In [ ]:
freq_matches = allmatches_df.matchtype.value_counts()
freq_matches

In [ ]:
allmatches_df2.matchtype.value_counts()

In [ ]:
allmatches_df2 = allmatches_df.copy()
allmatches_df2.sent = allmatches_df2.sent.str.replace(r'http\S+\s?','', regex=True)
allmatches_df2.sent = [str(s)[0:70] for s in allmatches_df2.sent]
for matchtype in freq_matches.index:
    print(matchtype+' ('+ str(freq_matches[matchtype]) +' hits)'+'\n----------')
    print(allmatches_df2[allmatches_df2.matchtype == matchtype].sent.value_counts()[:50])
    print('\n\n')
    with open(rootfold+'/flag_types.txt','a') as file:
        file.write(matchtype+' ('+ str(freq_matches[matchtype]) +' hits)'+'\n----------\n')
        file.write(str(allmatches_df2[allmatches_df2.matchtype == matchtype].sent.value_counts()[:25]))
        file.write('\n\n')

In [ ]:
allsents['flag2'] = 0
allsents.loc[pd.unique(allmatches_df.sent_index),'flag2'] = 1
allsents2 = allsents.copy()
allsents2.sent = allsents2.sent.str.replace(r'http\S+\s?','', regex=True)
allsents2.sent = [str(s)[:150] for s in allsents2.sent]
pd.pivot_table(allsents2, index = ['flag'], columns = ['flag2'], values = 'sent', aggfunc='count')

Posts that used to be flagged but are no longer flagged

In [ ]:
allsents2[(allsents2.flag==1) & (allsents2.flag2==0)].sent.value_counts()[:50]

Posts that are still not flagged

In [ ]:
allsents2[(allsents2.flag==0) & (allsents2.flag2==0)].sent.value_counts()[:50]

### Compute similarity

In [ ]:
sent_simil = {flgtype:pd.DataFrame(columns = ['sent','flag','simil']) for flgtype in flagtypes}
mean_flag_embeddings = dict()

for file in sentfiles:
    SENT = pd.read_csv(rootfold+'/../output/'+file)
    SENT.sent = SENT.sent.str.lower().str.replace('’','\'')
    print(file)
    for flgtype in flagtypes:
        print('Started:' + flgtype)
        s = SENT[SENT[flgtype]>0]
        initial_flags_embedings = embed(s.sent)
        mean_flag_embedding = tf.reduce_mean(embed(s[s.flag>0].sent), axis = 0)
        simildf = pd.DataFrame([(sent,flag,tf_simil(e, mean_flag_embedding)) 
                                     for sent,flag,e in zip(s.sent, s.flag, initial_flags_embedings)
                                     if tf_simil(e, mean_flag_embedding)>0.4], 
                                    columns = ['sent','flag','simil']).\
                        sort_values('simil', ascending=False)
        sent_simil[flgtype] = sent_simil[flgtype].append(simildf)
        print('Done:' + flgtype)
    print('\n\n')

### Most frequent similar that are not flags

In [ ]:
del sys.modules["functions_pos_match"]
import functions_pos_match as fpm

#### Disinformation

In [ ]:
s = sent_simil['disinformation']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:1000].index, n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:50].index, n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

#### Fake News

In [ ]:
s = sent_simil['fakenews']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:1000].index.str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:50].index.str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

#### BS

In [ ]:
s = sent_simil['bs']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:1000].index.str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:50].index.str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

#### Misleading / Clickbait

In [ ]:
s = sent_simil['misleading']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts().index[:1000].str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts().index[:100].str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

#### Unreliable (source)

In [ ]:
s = sent_simil['unreliable']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts().index[:1000].str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts().index[:1000].str.lower(), n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

#### Propaganda

In [ ]:
s = sent_simil['propaganda']
print(s[s.flag==0].sent.value_counts()[:20])
print('\n\n\n')
print(s[s.flag==0][:50])

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:1000].index, n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    if matched_keywords:
        print('%s | %s' % (sent, matched_keywords))

In [ ]:
for sent in fpm.nlp.pipe(s[s.flag==0].sent.value_counts()[:50].index, n_process=10, n_threads = 20, disable = ["ner"]):
    matched_keywords = sent[:]._.structurematched
    print('%s | %s' % (sent, matched_keywords))

### Non-REGEX

In [ ]:
comm_files = sorted(re.findall('COMM_2020_[0-9][0-9].csv', ' '.join(os.listdir(rootfold+'/../output'))))
comm_files

In [ ]:
comm_file = comm_files[len(comm_files)-1]
comm_generator = pd.read_csv(rootfold+'/../output/'+comm_file, chunksize=100000)
comm_file

In [ ]:
#for chunk_id, COMM in enumerate(comm_generator):

In [ ]:
COMM = next(comm_generator)
COMM = COMM[['subreddit','link_id','id','parent_id','body']]
COMM = COMM[~COMM.id.isin(existing_docids)]
COMM.body = COMM.body.astype(str)

In [ ]:
alldocs = list()
for doc_index, doc in enumerate(fpm.nlp.pipe(COMM.body, n_process=10, n_threads = 20, disable = ["tagger", "ner"])):
    sents = [s.text for s in doc.sents]
    for flgtype in flagtypes:
        list_05 = [(doc_index,sent,s,flgtype) \
                   for s,sent in zip([tf_simil(e, mean_flag_embeddings[flgtype]) for e in embed(sents)], sents) \
                   if s>0.5]
        if len(list_05)>0:
            alldocs += list_05
            print(list_05)

In [ ]:
alldocs_df = pd.DataFrame(alldocs, columns = ['id','sent','simil','flgtype'])
alldocs_df.sent.value_counts()

In [ ]:
comm_simil = {flgtype:pd.DataFrame(columns = ['sent','flag','simil']) for flgtype in flagtypes}
for flgtype in flagtypes:
        print('Started:' + flgtype)
        s = COMM.body
        initial_flags_embedings = embed(s.sent)
        mean_flag_embedding = tf.reduce_mean(embed(s[s.flag>0].sent), axis = 0)
        simildf = pd.DataFrame([(sent,flag,tf_simil(e, mean_flag_embedding)) 
                                     for sent,flag,e in zip(s.sent, s.flag, initial_flags_embedings)
                                     if tf_simil(e, mean_flag_embedding)>0.4], 
                                    columns = ['sent','flag','simil']).\
                        sort_values('simil', ascending=False)
        sent_simil[flgtype] = sent_simil[flgtype].append(simildf)

In [ ]:
fpm.regex_flag

In [ ]:
COMM[COMM.body.str.contains(fpm.regex_flag)]

In [ ]:
COMM[COMM.body.str.lower().str.contains(fpm.regex_flag)]

### Sarcasm and Irony - check

10 jul 2020

In [ ]:
comm_files = sorted(re.findall('COMM_2020_[0-9][0-9].csv', ' '.join(os.listdir(rootfold+'/../output'))))
comm_file = comm_files[len(comm_files)-1]
comm_generator = pd.read_csv(rootfold+'/../output/'+comm_file, chunksize=100000)
comm_file

In [ ]:
C

In [ ]:
sarcasm_regex = "\\\s\\b|\\/s\\b"  
irony_regex = '|'.join(["[*'\"]" + s for s in fpm.voc.attr.false + fpm.voc.attr.propaganda + fpm.voc.obj.disinfo])
joking_regex = '\\bjk\\b'
sarcasm_and_irony_regex = sarcasm_regex + '|' + irony_regex + '|' + joking_regex
sarcasm_and_irony_regex 

In [ ]:
artificial = pd.DataFrame({'body':['asd xxx','xxxdssa','Thanks Obama \s','Trump smart /s', 'Not related asd/s da',
                                   'I am sick of fake news',"I am sick of 'fake' news",'This "disinformation" must stop',
                                  '*fake* news is fake','idk jk','sadasdjkdsa']})
artificial['contains'] = artificial.body.str.contains(sarcasm_and_irony_regex)
artificial

In [ ]:
COMM = next(comm_generator)
COMM = COMM[['subreddit','link_id','id','parent_id','body']]
COMM.body = COMM.body.astype(str)

In [ ]:
test = COMM.body[:1000000]
n=0
for txt in test[test.str.lower().str.contains(sarcasm_and_irony_regex)]:
    if len(txt)<200:
        print(txt)
        print('\n\n')
    n+=1
print(n)

In [ ]:
COMM

In [ ]:
COMM.body.str.contains(fpm.bot_author_regex)

In [ ]:
COMM.body.str.contains('story')